In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import warnings
import palava
import json
import sys
import h5py

import os
import gc
from palava import settings
import matplotlib.patches as mpatches
import time 
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib
import argparse


/data/gpfs/projects/punim0614/sandeep_sk/Python_HPC/PALAVA-test/PALAVA/palava/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settings.seed = 0` to reproduce results from previous versions.
  self.seed = seed
/data/gpfs/projects/punim0614/sandeep_sk/Python_HPC/PALAVA-test/PALAVA/palava/_settings.py:70: UserWarning: Setting `dl_pin_memory_gpu_training` is deprecated in v1.0 and will be removed in v1.1. Please pass in `pin_memory` to the data loaders instead.
  self.dl_pin_memory_gpu_training = (


In [4]:
num_unann = int(40)

seed = 0


# Access the data directory
hyper = '0.5'

#string about experiment
str_ab_exp = 'fetal_erythropoesis'

palava_width = 200


print('palava_width', palava_width)

list_of_nonlin_factors = '20'

print('nonlin_facs', list_of_nonlin_factors)


print('string about experiment: ', str_ab_exp)

fdir = 'Out_files_and_results/'
fdir += str_ab_exp

fdir += '/' + 'fetal_erythropoesis_notebook_saved_model'
fdir += '/'

# Write all the figures to single pdf
adata = sc.read('/data/gpfs/projects/punim0614/sandeep_sk/Python_HPC/Simulated_analysis_palava_repo/fetal_erythropoesis_data_analysis/data/fetal_liver_erythroid_with_50H.h5ad')
pathways = [torch.tensor(i) for i in adata.uns['pathways_5000hvg']]

gene_names = adata.var

num_genes = len(adata.var)
pathway_names = adata.uns['pathway_names']
    
settings.seed = seed


SCVI_palava = palava.model.SCVI_palava

SCVI_palava.setup_anndata(adata, layer = 'counts')

pathways_bool = pathways



num_ann = len(pathway_names)

decoder_bias = True

if list_of_nonlin_factors == 'all':
    list_of_nonlin_factors = '-'.join([str(i) for i in range(num_ann + num_unann)])


if list_of_nonlin_factors == 'none':
    nonlin_facs = []
elif list_of_nonlin_factors == 'none_no_bias':
    nonlin_facs = []
    decoder_bias =False
else:
    nonlin_facs = [int(i) for i in list_of_nonlin_factors.split('-')]


nonlinear  = [0 for _ in range(num_ann + num_unann)]
print('Non linear factors')
for i in nonlin_facs:
    nonlinear[i] = 1

for i in range(num_ann):
    print(i, pathway_names[i])
    
plan_kwargs = {'lr' :  1e-4,'n_epochs_kl_warmup': 400,'n_steps_kl_warmup': None, 'max_kl_weight' : 1.0}

print(plan_kwargs)

lam = float(hyper)

lam_  = [lam] * num_ann # 50 hallmark + other genes
lam_ = lam_ + [lam / 2] * num_unann # unann

pathway_names_plot = [pathway_names[i].replace('_', ' ' ).capitalize() + ' ['+str(i)+']'  for i in range(len(pathway_names))] + ['Unannotated factor '+ str(i + 1) + ' ['+str(i+num_ann)+']'  for i in range(num_unann)]

scvi_palava = SCVI_palava(adata,  nonlinear  = nonlinear, n_annotated_latent = len(pathways_bool), n_unannotated_latent = num_unann, lam_lst = lam_ , other_n_latent = 40, pathways_bool = pathways_bool, dispersion = 'gene', palava_n_hidden = palava_width, palava_n_layers = 1,  n_layers = 3, use_batch_norm = 'none', use_layer_norm = 'none',  momentum_train = 0.1,  weight_decay_train = 0.0001, decoder_bias = decoder_bias, non_neg_decoder =True, just_out_weight_decay = True) #, gene_likelihood = 'nb')
print(scvi_palava)




palava_width 200
nonlin_facs 20
string about experiment:  fetal_erythropoesis


[rank: 0] Global seed set to 0
/data/gpfs/projects/punim0614/sandeep_sk/Python_HPC/anaconda3/envs/palava-env/lib/python3.10/abc.py:119: FutureWarning: SparseDataset is deprecated and will be removed in late 2024. It has been replaced by the public classes CSRDataset and CSCDataset.

For instance checks, use `isinstance(X, (anndata.experimental.CSRDataset, anndata.experimental.CSCDataset))` instead.

For creation, use `anndata.experimental.sparse_dataset(X)` instead.

  return _abc_instancecheck(cls, instance)


Non linear factors
0 HALLMARK_ADIPOGENESIS
1 HALLMARK_ALLOGRAFT_REJECTION
2 HALLMARK_ANDROGEN_RESPONSE
3 HALLMARK_ANGIOGENESIS
4 HALLMARK_APICAL_JUNCTION
5 HALLMARK_APICAL_SURFACE
6 HALLMARK_APOPTOSIS
7 HALLMARK_BILE_ACID_METABOLISM
8 HALLMARK_CHOLESTEROL_HOMEOSTASIS
9 HALLMARK_COAGULATION
10 HALLMARK_COMPLEMENT
11 HALLMARK_DNA_REPAIR
12 HALLMARK_E2F_TARGETS
13 HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION
14 HALLMARK_ESTROGEN_RESPONSE_EARLY
15 HALLMARK_ESTROGEN_RESPONSE_LATE
16 HALLMARK_FATTY_ACID_METABOLISM
17 HALLMARK_G2M_CHECKPOINT
18 HALLMARK_GLYCOLYSIS
19 HALLMARK_HEDGEHOG_SIGNALING
20 HALLMARK_HEME_METABOLISM
21 HALLMARK_HYPOXIA
22 HALLMARK_IL2_STAT5_SIGNALING
23 HALLMARK_IL6_JAK_STAT3_SIGNALING
24 HALLMARK_INFLAMMATORY_RESPONSE
25 HALLMARK_INTERFERON_ALPHA_RESPONSE
26 HALLMARK_INTERFERON_GAMMA_RESPONSE
27 HALLMARK_KRAS_SIGNALING_DN
28 HALLMARK_KRAS_SIGNALING_UP
29 HALLMARK_MITOTIC_SPINDLE
30 HALLMARK_MTORC1_SIGNALING
31 HALLMARK_MYC_TARGETS_V1
32 HALLMARK_MYC_TARGETS_V2
33 HALLMAR

PALAVA in SCVI framework with the following params: 
n_hidden: 128, lambda: [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 
0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 
0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 
0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 
0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], palava_n_hidden: 200 n_latent: 90, n_layers: 3, 
dropout_rate: 0.1, dispersion: gene, gene_likelihood: zinb, latent_distribution: normal
Training status: Not Trained
Model's adata is minified?: False

In [ ]:
scvi_palava.train(850, gradient_clip_val = 1e4, batch_size = 100, plan_kwargs = plan_kwargs )


/data/gpfs/projects/punim0614/sandeep_sk/Python_HPC/anaconda3/envs/palava-env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:168: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3.10 /home/ssanthoshkum/.local/lib/python3.10/site-pa ...
  rank_zero_warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/data/gpfs/projects/punim0614/sandeep_sk/Python_HPC/anaconda3/envs/palava-env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:168: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3.10 /home/ssanthoshkum/.local/lib/python3.10/site-pa ..

Epoch 6/850:   1%|          | 5/850 [01:17<3:39:10, 15.56s/it, v_num=1, train_loss_step=1.4e+3, train_loss_epoch=1.14e+3] 

In [ ]:

scvi_palava.save(fdir + '/latent_and_slope_data/scvi_model' )


factor_importance = scvi_palava.factor_importance(adata, num_batchs = 300)


#reverse order and top 10
index_asc_order = np.argsort(factor_importance)[::-1][:10]

plt.figure(figsize=(15, 5))
plt.bar(np.array(pathway_names_plot)[index_asc_order], factor_importance[index_asc_order] )
plt.xticks(rotation=90)
plt.savefig(fdir +'sorted factor importance num_batch 300.png', dpi = 150,bbox_inches='tight')

factor_importance_scores_dict = {'pathway_names_plot':pathway_names_plot,'factor_importance':factor_importance }

np.save(fdir+'factor_importance_scores_dict.npy', factor_importance_scores_dict, allow_pickle=True)


adata.obsm["X_scVI"] = scvi_palava.get_latent_representation()




latent = adata.obsm["X_scVI"][:,:len(pathways_bool)]



learned_activations = np.transpose(latent)

fig, ax = plt.subplots(figsize=(15, 7.5))

plt.imshow(learned_activations,   aspect = 'auto', cmap='bwr', interpolation = 'none', rasterized=True)
plt.title("Learned activation")
plt.xlabel('Cells')
plt.ylabel('Factors')
_ = plt.yticks([i for i in range(len(pathway_names_plot))], pathway_names_plot, fontsize = 7.5)
cbar = plt.colorbar()
plt.savefig(fdir+'image plot of latent space.png', dpi =150)
plt.close()
plt.clf()
gc.collect()

